<a href="https://colab.research.google.com/github/juanpajedrez/learn_rag_Huggingface/blob/main/MultiModal_RAG_Video.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 1  - Setup

In [ ]:
from google.colab import userdata
api_key = userdata.get('ai_agent_openai')

In [ ]:
%cd /content/drive/MyDrive/ai_agents_zero_to_mastery/RAG Bootcamp/RAG/Multimodal RAG

In [ ]:
from pathlib import Path
Path.cwd().exists()

# Step 2 -  Get the data

In [ ]:
# Define the video path
video_path = Path().cwd() / "decision-making-course.mp4"
video_path.exists()

# Step 3 - Extract Audio and Compress

In [ ]:
# Install libraries
!pip install -1 pydub
!apt-get install -q ffmpeg

In [ ]:
# Import libraries
import os
import subprocess
from pydub import AudioSegment

In [ ]:
# Define the audio path
audio_output_path = Path().cwd() / "audios" / "output.mp3"

In [ ]:
# Ensure the output path directory exists
output_dir = audio_output_path.parent
output_dir.mkdir(exist_ok=True, parents=True)

In [ ]:
# Ensure the output file has the correct extension
if not audio_output_path.suffix == ".mp3":
  audio_output_path = audio_output_path.with_suffix(".mp3")

In [ ]:
# Construct the ffmpeg command to extract the audio
command = [
    'ffmpeg',
    '-y', # overwrites if the audio exists
    '-i', str(video_path), # input file
    '-vn', # No video
    '-acodec', 'libmp3lame', # Audio codecs
    str(audio_output_path), # output file
]

In [ ]:
# Execute the command to extract the audio
subprocess.run(command, check=True)

In [ ]:
# Set the bitrate
# NOTE = Bitrate -> Frequency sample * bit depth * Number of channels (2 for stereo)
bitrate = "32k"

In [ ]:
# Set path for compressed audio
compressed_audio_path = Path().cwd() / "audios" / "output_compressed.mp3"

In [ ]:
# Construct the ffmpeg command to compress the audio
command = [
    'ffmpeg',
    '-y', # overwrites if the audio exists
    '-i', str(audio_output_path), # input file
    '-ab', bitrate, # bit rate
    str(compressed_audio_path), # output file
]

In [ ]:
subprocess.run(command, check=True)

# Step 4 - Transcribe Audio using OpenAI API

In [ ]:
!pip install -q openai

In [ ]:
# Library
from openai import OpenAI

In [ ]:
# Connect the script to the API
client = OpenAI(api_key=api_key)

In [ ]:
# Open the compressed file in binary mode
with open(compressed_audio_path, "rb") as file:
  # Use the Whisper model to transcribe
  transcript = client.audio.transcriptions.create(
      model="whisper-1",
      file=file,
  )

In [ ]:
# Inspect the transcript
transcript.text[0:500]

In [ ]:
# Define the path where the transcription will be saved
transcription_path = Path().cwd() / "transcripts" / "transcript.txt"

# Save the transcribed text to file
with open(transcription_path, "w") as file:
  file.write(transcript.text)

# Step 5 - Extract the frames.

In [ ]:
!pip install -q moviepy

In [ ]:
# Load library
from moviepy.editor import VideoFileClip

In [ ]:
# Degine output folder
output_folder = Path().cwd()/ "frames"
if not output_folder.exists():
  output_folder.mkdir(parents=True, exist_ok=True)

In [ ]:
# Load the video
video = VideoFileClip(str(video_path))

In [ ]:
# Extract the frames
from tqdm import tqdm
frame_paths = []

# Let's extract 60 frames per second, so a loooot of data.
# Therefore, we can limit the amount of data we need
interval = 10
total_frames = int((video.duration)/interval) + 1

for t in tqdm(range(0, int(video.duration), interval), desc = "Iterating across frames...", total = total_frames):
  frame_path = output_folder / f"frame_{t:04d}.png"

  # Save the frame at the specified time
  if not frame_path.exists():
    video.save_frame(str(frame_path), t)
  frame_paths.append(str(frame_path))

# Step 6 - Embedding Audio.

In [ ]:
# Import libraries
from transformers import CLIPProcessor, CLIPModel, CLIPTokenizer
import torch
import numpy as np

In [ ]:
# Load the Model, Processor and Tokenizer
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch32")

In [ ]:
# Get the transcribed text
with open(transcription_path, "r") as file:
  transcript_text = file.read()

In [ ]:
transcript_text

In [ ]:
# Tokenize the entire text
tokens = tokenizer(transcript_text,
                   return_tensors = "pt",
                   padding = True)
tokens = tokens['input_ids'][0]
print(f"The number of tokens is {len(tokens)}")

In [ ]:
# The CLIP model requires 77 tokens per chunk
max_tokens = 77
transcription_chunks = []

for i in range(0, len(tokens), max_tokens):
  chunk = tokens[i:i+max_tokens]
  transcription_chunks.append(chunk)

print(f"The number of chunks is: {len(transcription_chunks)}")

In [ ]:
# Text Embeddings -> Embedd the tokens in each chunk
text_embeddings = []

for chunk in tqdm(transcription_chunks, desc = "Getting text embeddings...", total = len(transcription_chunks)):
  # Ensure the chunk is in the correct shape
  inputs = {"input_ids": chunk.unsqueeze(0)}

  # Get the text embedding
  with torch.inference_mode():
    text_embedding = model.get_text_features(**inputs)
    text_embeddings.append(text_embedding['pooler_output'].cpu().numpy().flatten())

# Convert the list of embeddings to a numpy array
text_embedding_np = np.array(text_embeddings)

In [ ]:
# Print the dimensions and embeddings necessary
print(f"The torch text single embedding shape is: {transcription_chunks[0].shape}, dtype: {transcription_chunks[0].dtype}")
print(f"The numpy text embeddings shape is: {text_embedding_np.shape}, dtype: {text_embedding_np.dtype}")

# Step 7 -  embedding the images

In [ ]:
from PIL import Image

In [ ]:
# Embedd the images
frames_folder_path = Path().cwd() / "frames"
image_embeddings = []

# Let's get all filenames
all_files_paths = list(frames_folder_path.glob("*"))

for frame_path in tqdm(all_files_paths, desc = "Getting the image embeddings...", total = len(all_files_paths)):
  if frame_path.suffix == ".png":
    # Load and preprocess the image
    image = Image.open(frame_path)
    inputs = processor(images = image, return_tensors = "pt")

    # Pass it to the model to obtain image embeddings
    with torch.inference_mode():
      image_embedding = model.get_image_features(**inputs)
      image_embeddings.append(image_embedding['pooler_output'].cpu().numpy().flatten())

# Cast them into numpy
image_embedding_np = np.array(image_embeddings)

In [ ]:
# Print the dimensions and embeddings necessary
test_img = Image.open(all_files_paths[0])

# Print all of the dimensions and types of image embeddings
print(f"The image torch single embedding shape is: {test_img.size}, type: {type(test_img)}")
print(f"The image numpy embedding shape is: {image_embedding_np.shape}, dtype: {image_embedding_np.dtype}")

# Step 8 - Contrastive Learning

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import random

In [ ]:
# Calculate the cosine similarity matrix
similarity_matrix = cosine_similarity(text_embedding_np, image_embedding_np)

In [ ]:
similarity_matrix.shape

In [ ]:
# Retrieve the top-k similar images for each text chunk
top_k = 5
for i , text_chunk in enumerate(similarity_matrix):
  similar_indices = text_chunk.argsort()[-top_k:][::-1] # Starting from the last one to the first one
  print(f"Top {top_k} images for each chunk: {i}: {similar_indices}")

In [ ]:
# Set a random seed for reproducibility
random.seed(1502)

# Select 5 random text chunk indices
random_text_indices = random.sample(range(len(text_embedding_np)), 5)
print(f"Random text chunk indices: {random_text_indices}")

In [ ]:
# Find the 3 most similar images for each text
text_to_images_similarities = []
for idx in random_text_indices:
  similar_images = similarity_matrix[idx].argsort()[-3:][::-1] # Get the top 3 and invert everything
  text_to_images_similarities.append(similar_images)

In [ ]:
# Data visualization of the text and images
for i, text_idx in enumerate(random_text_indices):
  plt.figure(figsize = (10, 8))

  # Displaying the text chunk with a print
  print(f"Text chunk: {i + 1}: {" ".join(
      [tokenizer.decode([token]) for token in transcription_chunks[text_idx]])
  }")
  for j, image_idx in enumerate(text_to_images_similarities[i]):
    image = Image.open(all_files_paths[image_idx])
    plt.subplot(1, 3, j + 1)
    plt.imshow(image)
    plt.axis("off")
  plt.show()


# Step 9 - Retrieval System

In [ ]:
# Let's define a query
query = "Which cognitive biases are discussed?"

In [ ]:
# Tokenizing the query
query_tokens = tokenizer(query,
                         return_tensors = "pt",
                         padding=True)["input_ids"]

In [ ]:
# Generate the query embeddings in the join embedding space
# Use the clip model
with torch.inference_mode():
  query_embedding = model.get_text_features(
      input_ids = query_tokens
  )['pooler_output'].cpu().numpy().flatten()
print(f"The shape of the query embedding is: {query_embedding.shape}")

In [ ]:
# Compute the cosine similarity between the query and the transcripts
cos_similarities_query_text = cosine_similarity([query_embedding], text_embedding_np)[0]

In [ ]:
# Define how many chunks we want
top_k_text = 10

# Retrieve the indices of the top-k most similar text-chunks
top_k_text_indices = cos_similarities_query_text.argsort()[-top_k_text:][::-1]
top_k_text_indices

In [ ]:
# Retrieve the closes images for each text chunk
top_k_images_indices = []
number_of_images_per_chunk = 2

for idx in top_k_text_indices:
  similar_images = similarity_matrix[idx].argsort()[-number_of_images_per_chunk:][::-1]
  top_k_images_indices.append(similar_images)

# Remove any duplicates and limit to top k images
top_k_images_indices = list(set([item for sublist in top_k_images_indices for item in sublist]))
print(f"The total images indices are {len(top_k_images_indices)}")

# Step 10 - Generation System

In [ ]:
import base64

In [ ]:
# Combining the retrieved text chunks
retrieved_text = []
for idx in top_k_text_indices:
  retrieved_text.append(tokenizer.decode(transcription_chunks[idx]))
retrieved_text = " ".join(retrieved_text)
retrieved_text

In [ ]:
# Convert the images and append them
base64frames = []
for idx in top_k_images_indices:
  image_path = all_files_paths[idx]
  with open(image_path, "rb") as image_file:
    base64_image = base64.b64encode(image_file.read()).decode("utf-8")
    base64frames.append(base64_image)

In [ ]:
# Define the mode answer system prompt
MODEL = "gpt-5.4-mini"
system_prompt = """
You are an expert teacher that summarizes visual and transcribed content
"""

In [ ]:
# Prepare tge yser nessage content
user_message_content = [
    "The are the frames from the video",
    *map(lambda x: {"type": "image_url",
                      "image_url": {"url": f"data:image/jpg;base64, {x}",
                                    "detail": "high"}},
         base64frames),
    {"type": "text",
     "text": retrieved_text}
]

In [ ]:
# Call the OpenAI API to generate a summary
response = client.chat.completions.create(
    model = MODEL,
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message_content},
    ],
    temperature = 0.3,
)

In [ ]:
import IPython
from IPython.display import display, Markdown

In [ ]:
# Generate the response
generated_response = response.choices[0].message.content
display(Markdown(generated_response))